<a href="https://colab.research.google.com/github/betulbilhan2/LLM-Augmentation-Fidelity/blob/main/NB07_BERTurk_FineTuning_Normal_Part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 07: BERTurk Fine-Tuning (80 Runs)

**Hedef:**
1. 2 Resource Level (`low`, `normal`) x 4 Senaryo (E0, E1, E2, E3) x 10 Seed = 80 koşumu tamamlamak.
2. Colab ortamı için kesintilere karşı dayanıklı (resumable) döngü kurmak.
3. Disk ve GPU belleği şişmesini önlemek için her koşum sonrası model ağırlıklarını silip belleği temizlemek.
4. Metrikleri (macro-F1 ve sınıf bazlı F1) kaydedip tek bir CSV'ye eklemek.

In [ ]:
!pip install -q transformers datasets evaluate scikit-learn pandas pyyaml

import os
import gc
import json
import yaml
import torch
import shutil
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    set_seed
)
import evaluate
from sklearn.metrics import f1_score, classification_report

# Colab mount
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/tr_augmentation_project'
    IN_COLAB = True
except:
    BASE_DIR = 'C:/Users/btlbi/OneDrive/Masaüstü/TR Veri arttırımı'
    IN_COLAB = False
    print("Colab ortamı bulunamadı, yerel dizin kullanılıyor:", BASE_DIR)

# Çıktı dizinleri
RESULTS_DIR = os.path.join(BASE_DIR, '07_results')
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'temp_checkpoints')
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Dizinler hazır.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
Mounted at /content/drive
Dizinler hazır.


In [ ]:
# Load Config
config_path = os.path.join(BASE_DIR, 'configs', 'experiment_config.yaml')
if os.path.exists(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
else:
    # Fallback default if config missing
    config = {
        'seeds': {'data_seeds': list(range(10)), 'model_seed': 42},
        'model': {'name': 'dbmdz/bert-base-turkish-cased', 'max_length': 128},
        'training': {'early_stopping_patience': 3}
    }

DATA_SEEDS = config['seeds']['data_seeds']
MODEL_SEED = config['seeds']['model_seed']
MODEL_NAME = config['model']['name']
MAX_LEN = config['model']['max_length']
RESOURCE_LEVELS = ['low', 'normal']
SCENARIOS = ['E0', 'E1', 'E2', 'E3']

print(f"Model: {MODEL_NAME}\nSeeds: {DATA_SEEDS}\nLevels: {RESOURCE_LEVELS}")

Model: dbmdz/bert-base-turkish-cased
Seeds: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Levels: ['low', 'normal']


In [ ]:
# Yardımcı Fonksiyonlar

def get_train_data(level, scenario, seed):
    # E0: Original-only
    train_orig_path = os.path.join(BASE_DIR, '01_splits', level, f'seed_{seed}', 'train_seed.csv')

    # Hata ayıklama (Drive senkronizasyon kontrolü)
    if not os.path.exists(train_orig_path):
        print(f"\n[HATA] Dosya bulunamadı: {train_orig_path}")
        parent_dir = os.path.join(BASE_DIR, '01_splits', level)
        if os.path.exists(parent_dir):
            print(f"Bunun yerine {parent_dir} içindeki mevcut klasörler:")
            print(os.listdir(parent_dir))
        else:
            print(f"{parent_dir} dizini komple yok!")

    df_train = pd.read_csv(train_orig_path)

    if scenario == 'E0':
        return df_train

    elif scenario == 'E1':
        # Duplication Control
        pool_path = os.path.join(BASE_DIR, '02_augmented', level, 'duplication_control', f'seed_{seed}', 'pool.csv')
        df_aug = pd.read_csv(pool_path)
        return pd.concat([df_train, df_aug], ignore_index=True)

    elif scenario == 'E2':
        # Original + BT
        pool_path = os.path.join(BASE_DIR, '05_balanced', level, 'backtranslation', f'seed_{seed}', 'pool_balanced.csv')
        df_aug = pd.read_csv(pool_path)
        return pd.concat([df_train, df_aug], ignore_index=True)

    elif scenario == 'E3':
        # Original + LLM
        pool_path = os.path.join(BASE_DIR, '05_balanced', level, 'llm_paraphrase', f'seed_{seed}', 'pool_balanced.csv')
        df_aug = pd.read_csv(pool_path)
        return pd.concat([df_train, df_aug], ignore_index=True)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    macro_f1 = f1_score(labels, predictions, average='macro')
    # Sınıf bazlı F1 (0: Negatif, 1: Nötr, 2: Pozitif varsayıyoruz, duruma göre etiket sıralaması değişebilir)
    class_f1 = f1_score(labels, predictions, average=None)

    res = {'macro_f1': macro_f1}
    for i, f1 in enumerate(class_f1):
        res[f'f1_class_{i}'] = f1
    return res

In [ ]:
# Veri Yükleme ve Tokenization Hazırlığı
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def prepare_dataset(df, split_name="unknown"):
    label_col = 'label' if 'label' in df.columns else 'sentiment'

    raw_uniques = df[label_col].unique()

    # Ensure all labels map correctly without silent NaNs
    if pd.api.types.is_numeric_dtype(df[label_col]) or all(str(x).isdigit() for x in raw_uniques):
        df['label'] = df[label_col].astype(int)
    else:
        label_mapping = {'Negative': 0, 'Notr': 1, 'Positive': 2, 'negative': 0, 'neutral': 1, 'positive': 2}
        mapped_labels = df[label_col].map(label_mapping)
        n_missing = mapped_labels.isna().sum()

        if n_missing > 0:
            raise ValueError(f"[{split_name}] Label mapping error! Unmatched labels found. Raw uniques: {raw_uniques}. {n_missing} NaN values produced.")

        df['label'] = mapped_labels.astype(int)

    # Boundary check (Semantic protection)
    final_labels = set(df['label'].unique())
    if not final_labels.issubset({0, 1, 2}):
        raise ValueError(f"[{split_name}] Beklenmeyen etiket degerleri bulundu (0, 1, 2 disinda): {final_labels - {0, 1, 2}}")

    ds = Dataset.from_pandas(df[['text', 'label']])
    return ds.map(
        lambda x: tokenizer(x['text'], truncation=True, padding='max_length', max_length=MAX_LEN),
        batched=True
    )

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/251k [00:00<?, ?B/s]

In [ ]:
# 80 Koşumluk Ana Döngü

import time

# Paralel Çalıştırma (Parallel Execution) Ayarları
# Eğer birden fazla Colab hesabında bölecekseniz, bu listeleri daraltın.
# Örnek 1. Hesap: RESOURCE_LEVELS = ['low'], SCENARIOS = ['E0', 'E1', 'E2', 'E3']
# Örnek 2. Hesap: RESOURCE_LEVELS = ['normal'], SCENARIOS = ['E0', 'E1']
# Örnek 3. Hesap: RESOURCE_LEVELS = ['normal'], SCENARIOS = ['E2', 'E3']

RESOURCE_LEVELS = [ 'normal']
SCENARIOS = ['E0', 'E1']

# Çıktı CSV dosyasının ismini çakışmaları önlemek için çalışılan ayarlara göre dinamik yapıyoruz
csv_suffix = "_".join(RESOURCE_LEVELS) + "_" + "_".join(SCENARIOS)
results_csv_path = os.path.join(RESULTS_DIR, f'summary_{csv_suffix}.csv')

# Eğer varsa mevcut CSV'yi yükle, yoksa başlıkları hazırla
if not os.path.exists(results_csv_path):
    with open(results_csv_path, 'w', encoding='utf-8') as f:
        f.write("resource_level,scenario,seed,macro_f1,f1_class_0,f1_class_1,f1_class_2,train_time_sec\n")

# TEST MODU: Tüm döngüyü çalıştırmadan önce 1 tur denemek için True yapın
TEST_RUN = False
if TEST_RUN:
    RESOURCE_LEVELS = ['low']
    SCENARIOS = ['E0']
    DATA_SEEDS = [0]
    results_csv_path = os.path.join(RESULTS_DIR, 'summary_test_run.csv')
    print("!!! TEST RUN AKTIF: Sadece low-E0-seed0 çalışacak !!!")

for level in RESOURCE_LEVELS:
    for scenario in SCENARIOS:
        for seed in DATA_SEEDS:
            run_id = f"{level}_{scenario}_seed{seed}"
            json_out_path = os.path.join(RESULTS_DIR, f"{run_id}.json")

            # 1. Colab'ta kesinti/timeout dayanıklılığı (zaten yapıldıysa atla)
            if os.path.exists(json_out_path):
                print(f"[{run_id}] Zaten tamamlanmış, atlanıyor...")
                continue

            print(f"\n>>> Başlıyor: {run_id}")
            start_time = time.time()

            # Seed sabitlemesi (Model ve DataLoader için sızıntı önleme)
            set_seed(MODEL_SEED)

            run_ckpt_dir = None
            try:
                # Verileri Hazırla
                df_train = get_train_data(level, scenario, seed)
                val_path = os.path.join(BASE_DIR, '01_splits', 'fixed', 'validation.csv')
                df_val = pd.read_csv(val_path)

                # UYARI ÇÖZÜMÜ: Val seti çok büyükse (22k), her epoch sonu eval yapmak 2 dakika sürer.
                # Early stopping için 1500 örnek (sınıf başı ~500) fazlasıyla yeterlidir.
                if len(df_val) > 1500:
                    df_val = df_val.sample(n=1500, random_state=42)

                # Test seti
                test_path = os.path.join(BASE_DIR, '01_splits', 'fixed', 'test.csv')
                if os.path.exists(test_path):
                    current_df_test = pd.read_csv(test_path)
                else:
                    print(f"\n[HATA] Test dosyası bulunamadı: {test_path}")
                    parent_dir = os.path.join(BASE_DIR, '01_splits', 'fixed')
                    print(os.listdir(parent_dir) if os.path.exists(parent_dir) else f"{parent_dir} yok!")
                    raise FileNotFoundError(f"Test seti bulunamadı, val ile sessizce devam edilemez: {test_path}")

                # Veri sızıntısı kontrolü
                if current_df_test.equals(df_val):
                    raise ValueError("KRITIK HATA: Test seti ile Validation seti birebir aynı! Veri sızıntısı (leakage) var.")

                train_ds = prepare_dataset(df_train, "train")
                val_ds = prepare_dataset(df_val, "val")
                test_ds = prepare_dataset(current_df_test, "test")

                num_labels = len(df_train['label' if 'label' in df_train.columns else 'sentiment'].unique())
                model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

                # Checkpoint dir for this specific run
                run_ckpt_dir = os.path.join(CHECKPOINT_DIR, run_id)

                # Eğitim argümanları (Eval ve Save stratejisi uyumlu, epoch bazlı)
                # Data_seed sadece datayı çektiğimiz csv'yi etkiler, geri kalanı MODEL_SEED'e bağlı.
                training_args = TrainingArguments(
                    output_dir=run_ckpt_dir,
                    eval_strategy="epoch",
                    save_strategy="epoch",
                    learning_rate=2e-5,
                    per_device_train_batch_size=16,
                    per_device_eval_batch_size=32,
                    num_train_epochs=15 if level == 'low' else 5, # low kaynakta daha çok epoch gerekebilir
                    weight_decay=0.01,
                    load_best_model_at_end=True,
                    metric_for_best_model="macro_f1",
                    save_total_limit=1,
                    seed=MODEL_SEED,
                    data_seed=MODEL_SEED,
                    logging_strategy="epoch",
                    report_to="none"
                )

                trainer = Trainer(
                    model=model,
                    args=training_args,
                    train_dataset=train_ds,
                    eval_dataset=val_ds,
                    compute_metrics=compute_metrics,
                    callbacks=[EarlyStoppingCallback(early_stopping_patience=config['training']['early_stopping_patience'])]
                )

                # Modeli Eğit
                trainer.train()

                # Test Seti Üzerinde Değerlendir (Tek geçiş: trainer.predict ile hem tahminleri hem metrikleri alıyoruz)
                preds = trainer.predict(test_ds)
                test_results = preds.metrics
                pred_labels = np.argmax(preds.predictions, axis=-1)

                train_time = time.time() - start_time

                # Metrikleri JSON'a kaydet (Ayrıntılı ve tahminlerle birlikte)
                run_metrics = {
                    'run_id': run_id,
                    'level': level,
                    'scenario': scenario,
                    'seed': seed,
                    'test_results': test_results,
                    'predictions': pred_labels.tolist(),
                    'train_time_sec': train_time
                }

                with open(json_out_path, 'w', encoding='utf-8') as f:
                    json.dump(run_metrics, f, indent=4)

                # CSV'ye Append (Hemen kaydet - trainer.predict varsayılan olarak 'test_' öneki kullanır)
                macro = test_results.get('test_macro_f1', 0)
                f1_0 = test_results.get('test_f1_class_0', 0)
                f1_1 = test_results.get('test_f1_class_1', 0)
                f1_2 = test_results.get('test_f1_class_2', 0)

                with open(results_csv_path, 'a', encoding='utf-8') as f:
                    f.write(f"{level},{scenario},{seed},{macro},{f1_0},{f1_1},{f1_2},{train_time}\n")

                print(f"[{run_id}] Tamamlandı. Macro-F1: {macro:.4f}, Süre: {train_time:.1f}s")

            except Exception as e:
                print(f"[{run_id}] HATA OLUŞTU: {e}")
                with open(os.path.join(RESULTS_DIR, 'error_log.txt'), 'a') as ef:
                    ef.write(f"{run_id} failed: {str(e)}\n")

            finally:
                # 2. Disk Temizliği: Model ağırlıklarını diskte tutma (kota aşımını önle)
                if run_ckpt_dir and os.path.exists(run_ckpt_dir):
                    shutil.rmtree(run_ckpt_dir)

                # 3. Bellek Temizliği: GPU'nun şişmesini önle
                try:
                    del model
                    del trainer
                except:
                    pass
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

print("\n--- Tüm Çalışmalar Tamamlandı veya Atlandı ---")


>>> Başlıyor: normal_E0_seed0


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/22034 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  445MB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[normal_E1_seed9] Tamamlandı. Macro-F1: 0.8430, Süre: 349.6s

--- Tüm Çalışmalar Tamamlandı veya Atlandı ---


## Sağlık Kontrolü Sonrası Not
Eğer `TEST_RUN = True` ile denediyseniz ve her şey yolundaysa (Drive'da `.json` ve CSV güncellendiyse), `TEST_RUN = False` yapıp **Tümünü Çalıştır (Run All)** ile tam döngüyü başlatabilirsiniz.
Colab kapanırsa, tekrar açıp baştan çalıştırın; zaten tamamlanmış tohumlar hızlıca `continue` ile atlanıp kaldığı yerden devam edecektir.